# March Madness 2025 Predictions
Complete workflow for NCAA basketball predictions

## 1. Initial Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [2]:
# Configure display options
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

In [3]:
# Memory optimization
dtypes = {
    'Season': 'int16',
    'TeamID': 'int16',
    'WTeamID': 'int16', 
    'LTeamID': 'int16'
}

## 2. Data Loading

In [4]:
# Load team lists
m_teams = pd.read_csv('data/MTeams.csv')['TeamID'].unique()
w_teams = pd.read_csv('data/WTeams.csv')['TeamID'].unique()

# Load regular season data
m_reg = pd.read_csv('data/MRegularSeasonCompactResults.csv', dtype=dtypes)
w_reg = pd.read_csv('data/WRegularSeasonCompactResults.csv', dtype=dtypes)

# Load tournament data
m_tourney = pd.read_csv('data/MNCAATourneyCompactResults.csv', dtype=dtypes)
w_tourney = pd.read_csv('data/WNCAATourneyCompactResults.csv', dtype=dtypes)

# Load seeds
m_seeds = pd.read_csv('data/MNCAATourneySeeds.csv', dtype=dtypes)
w_seeds = pd.read_csv('data/WNCAATourneySeeds.csv', dtype=dtypes)

## 3. Data Preprocessing

In [6]:
def process_seeds(seeds_df, teams):
    """Robust seed processing with multiple format handling"""
    # Extract numerical portion from seed strings
    seeds_df['Seed'] = (
        seeds_df['Seed']
        .str.extract(r'(\d+)', expand=False)  # Get any numeric part
        .fillna('16')  # Fill missing with worst seed
        .astype(int)
    )
    
    # Create complete grid of all seasons and teams
    all_seasons = pd.DataFrame({'Season': range(1985, 2026)})
    full_grid = all_seasons.merge(
        pd.DataFrame({'TeamID': teams}), 
        how='cross'
    )
    
    # Merge and fill remaining missing seeds
    merged = full_grid.merge(
        seeds_df,
        on=['Season', 'TeamID'],
        how='left'
    ).fillna({'Seed': 16})
    
    return merged.astype({'Seed': 'int8'})

# Load raw seed data
m_seeds_raw = pd.read_csv('data/MNCAATourneySeeds.csv')
w_seeds_raw = pd.read_csv('data/WNCAATourneySeeds.csv')

# Process seeds
m_seeds_complete = process_seeds(m_seeds_raw, m_teams)
w_seeds_complete = process_seeds(w_seeds_raw, w_teams)

# Combine all seeds
all_seeds = pd.concat([m_seeds_complete, w_seeds_complete])

In [ ]:
def process_seeds(seeds_df, gender):
    """
    Processes seeds with multiple format handling:
    - Men's format: W01, X11, Y16a, Z16b
    - Women's format: A01, B02, C15a, D16b
    - Special cases: 16a, 16b (play-in games)
    """
    # Extract numerical portion and play-in status
    seeds_df['Seed'] = (
        seeds_df['Seed']
        .str.replace(r'^[A-Za-z]*', '', regex=True)  # Remove prefix
        .str.extract(r'(\d+)', expand=False)          # Get numeric part
        .fillna('16')                                 # Fill missing
        .astype(int)                                  # Convert to integer
    )
    
    # Create complete grid of all seasons and teams
    all_seasons = pd.DataFrame({'Season': range(1985, 2026)})
    teams = m_teams if gender == 'M' else w_teams
    full_grid = all_seasons.merge(
        pd.DataFrame({'TeamID': teams}), 
        how='cross'
    )
    
    # Merge and fill remaining missing seeds
    merged = full_grid.merge(
        seeds_df,
        on=['Season', 'TeamID'],
        how='left'
    ).fillna({'Seed': 16})
    
    return merged.astype({'Seed': 'int8'})

# Process seeds with gender-specific handling
m_seeds = pd.read_csv('data/MNCAATourneySeeds.csv')
w_seeds = pd.read_csv('data/WNCAATourneySeeds.csv')

m_seeds_clean = process_seeds(m_seeds, 'M')
w_seeds_clean = process_seeds(w_seeds, 'W')

# Combine all seeds
all_seeds = pd.concat([m_seeds_clean, w_seeds_clean])

: 

In [ ]:


# %% [markdown]
# ## 4. Feature Engineering
# %%
def calculate_team_stats(reg_data):
    """Calculate 3-year rolling averages"""
    stats = reg_data.groupby(['Season', 'WTeamID']).agg({
        'WScore': ['mean', 'std'],
        'LScore': ['mean', 'std']
    })
    stats.columns = ['Off_Avg', 'Off_Std', 'Def_Avg', 'Def_Std']
    return stats.groupby(level=1).transform(lambda x: x.rolling(3, 1).mean())

# Calculate stats for both genders
m_stats = calculate_team_stats(m_reg).reset_index()
w_stats = calculate_team_stats(w_reg).reset_index()

# Combine stats
all_stats = pd.concat([m_stats, w_stats])

# %% [markdown]
# ## 5. Prepare Training Data
# %%
# Create training dataset from historical results
train_data = pd.concat([m_tourney, w_tourney])
train_data['Team1'] = np.minimum(train_data['WTeamID'], train_data['LTeamID'])
train_data['Team2'] = np.maximum(train_data['WTeamID'], train_data['LTeamID'])
train_data['Target'] = (train_data['WTeamID'] == train_data['Team1']).astype(int)

# Merge features
train_features = train_data.merge(
    all_seeds, 
    left_on=['Season', 'Team1'],
    right_on=['Season', 'TeamID']
).merge(
    all_seeds,
    left_on=['Season', 'Team2'],
    right_on=['Season', 'TeamID'],
    suffixes=('_1', '_2')
).merge(
    all_stats,
    left_on=['Season', 'Team1'],
    right_on=['Season', 'WTeamID']
).merge(
    all_stats,
    left_on=['Season', 'Team2'],
    right_on=['Season', 'WTeamID'],
    suffixes=('_1', '_2')
)

# Create final features
X = train_features[[
    'Seed_1', 'Seed_2', 
    'Off_Avg_1', 'Off_Avg_2',
    'Def_Avg_1', 'Def_Avg_2'
]].copy()
X['Seed_Diff'] = X['Seed_1'] - X['Seed_2']
X['Off_Diff'] = X['Off_Avg_1'] - X['Off_Avg_2']
X['Def_Diff'] = X['Def_Avg_1'] - X['Def_Avg_2']
X = X[['Seed_Diff', 'Off_Diff', 'Def_Diff']]

y = train_features['Target']

# %% [markdown]
# ## 6. Model Training
# %%
# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Train model
model = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.01,
    max_depth=3,
    random_state=42
)
model.fit(X_train_scaled, y_train)

# Validate
val_preds = model.predict_proba(X_val_scaled)[:, 1]
print(f"Validation ROC AUC: {roc_auc_score(y_val, val_preds):.4f}")

# %% [markdown]
# ## 7. Prepare Submission Data
# %%
# Load submission template
submission = pd.read_csv('data/SampleSubmissionStage1.csv')

# Parse IDs
submission[['Season', 'Team1', 'Team2']] = (
    submission['ID'].str.split('_', expand=True)
submission = submission.astype({
    'Season': 'int16',
    'Team1': 'int16',
    'Team2': 'int16'
})

# %% [markdown]
# ## 8. Generate Predictions
# %%
# Merge features for submission
sub_features = submission.merge(
    all_seeds, 
    left_on=['Season', 'Team1'],
    right_on=['Season', 'TeamID']
).merge(
    all_seeds,
    left_on=['Season', 'Team2'],
    right_on=['Season', 'TeamID'],
    suffixes=('_1', '_2')
).merge(
    all_stats,
    left_on=['Season', 'Team1'],
    right_on=['Season', 'WTeamID']
).merge(
    all_stats,
    left_on=['Season', 'Team2'],
    right_on=['Season', 'WTeamID'],
    suffixes=('_1', '_2')
)

# Create final features
X_sub = sub_features[[
    'Seed_1', 'Seed_2', 
    'Off_Avg_1', 'Off_Avg_2',
    'Def_Avg_1', 'Def_Avg_2'
]].copy()
X_sub['Seed_Diff'] = X_sub['Seed_1'] - X_sub['Seed_2']
X_sub['Off_Diff'] = X_sub['Off_Avg_1'] - X_sub['Off_Avg_2']
X_sub['Def_Diff'] = X_sub['Def_Avg_1'] - X_sub['Def_Avg_2']
X_sub = X_sub[['Seed_Diff', 'Off_Diff', 'Def_Diff']]

# Scale and predict
X_sub_scaled = scaler.transform(X_sub)
submission['Pred'] = model.predict_proba(X_sub_scaled)[:, 1]

# %% [markdown]
# ## 9. Save Results
# %%
# Save final submission
submission[['ID', 'Pred']].to_csv('2025_predictions.csv', index=False)

# Validation checks
print("Submission Summary:")
print(f"Total predictions: {len(submission):,}")
print(f"Prediction range: {submission['Pred'].min():.4f} - {submission['Pred'].max():.4f}")
print("\nSample predictions:")
print(submission.head())

SyntaxError: invalid syntax (2088390107.py, line 163)